# Generación y versionado del dataset final

Este notebook es el checkpoint de Persona D. Carga el candidato, confirma que no hay correcciones adicionales aprobadas, ejecuta las siete validaciones, verifica los invariantes de entrega y exporta de forma determinista el único CSV limpio final, versión `v1.0.0`.

## Configuración, esquema y metadatos

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src import catalogos
from src.finalizacion import (
    FECHA_EXTRACCION, FUENTE_DATOS, VERSION_DATASET,
    exportar_dataset_final, validar_codebook, validar_invariantes_finales,
)
from src.validadores import COLUMNAS_REQUERIDAS, cargar_csv_para_validacion, ejecutar_validaciones, resumen_validaciones

In [2]:
RUTA_CANDIDATO = ROOT / "data" / "processed" / "establecimientos_limpios_candidato.csv"
RUTA_FINAL = ROOT / "data" / "processed" / "establecimientos_clean.csv"
RUTA_CODEBOOK = ROOT / "codebook.md"

print("Versión:", VERSION_DATASET)
print("Fecha de extracción:", FECHA_EXTRACCION)
print("Fuente:", FUENTE_DATOS)

Versión: v1.0.0
Fecha de extracción: 2026-07-18
Fuente: MINEDUC, Buscador de establecimientos educativos


## Checkpoint y correcciones adicionales

In [3]:
correcciones_adicionales = pd.DataFrame(
    columns=["CODIGO", "VARIABLE", "ANTES", "DESPUES", "JUSTIFICACION"]
)
assert correcciones_adicionales.empty
print("Correcciones adicionales aprobadas en el checkpoint: 0")
print("El candidato se promueve sin alterar valores; solo se ordena por CODIGO.")

Correcciones adicionales aprobadas en el checkpoint: 0
El candidato se promueve sin alterar valores; solo se ordena por CODIGO.


## Siete validaciones previas a la escritura

In [4]:
df_candidato = cargar_csv_para_validacion(RUTA_CANDIDATO)
resultados_previos = ejecutar_validaciones(df_candidato)
resumen_validaciones(resultados_previos)

,Prueba,Estado,Errores,Detalle
0,Duplicados exactos,APROBADO,0,No deben existir filas idénticas en todas las ...
1,Espacios en textos,APROBADO,0,"Los textos no deben tener espacios al inicio, ..."
2,Formato de teléfonos,APROBADO,0,"Cada teléfono debe tener 8 dígitos, iniciar en..."
3,Geografía oficial,APROBADO,0,Departamento y municipio deben pertenecer al c...
4,Esquema y tipos,APROBADO,0,El candidato debe contener las 18 columnas esp...
5,Categorías equivalentes,APROBADO,0,No deben coexistir categorías que solo difiera...
6,Valores inválidos diagnosticados,APROBADO,0,"Códigos, distritos, dominios, faltantes obliga..."


In [5]:
fallos_previos = [r for r in resultados_previos if not r.aprobada]
if fallos_previos:
    raise RuntimeError({r.prueba: r.ejemplos for r in fallos_previos})
assert len(resultados_previos) == 7
print("VALIDACIÓN PREVIA APROBADA: 7 de 7.")

VALIDACIÓN PREVIA APROBADA: 7 de 7.


## Consistencia del esquema y del libro de códigos

In [6]:
assert list(df_candidato.columns) == COLUMNAS_REQUERIDAS
assert all(str(tipo) == "string" for tipo in df_candidato.dtypes)
validar_codebook(RUTA_CODEBOOK, list(df_candidato.columns))
print("Esquema explícito:", {columna: "string" for columna in COLUMNAS_REQUERIDAS})
print("Codebook: 18 de 18 variables, nueve campos completos por variable.")

Esquema explícito: {'CODIGO': 'string', 'DISTRITO': 'string', 'DEPARTAMENTO': 'string', 'MUNICIPIO': 'string', 'ZONA_CAPITAL': 'string', 'ESTABLECIMIENTO': 'string', 'DIRECCION': 'string', 'TELEFONO': 'string', 'SUPERVISOR': 'string', 'DIRECTOR': 'string', 'NIVEL': 'string', 'SECTOR': 'string', 'AREA': 'string', 'STATUS': 'string', 'MODALIDAD': 'string', 'JORNADA': 'string', 'PLAN': 'string', 'DEPARTAMENTAL': 'string'}
Codebook: 18 de 18 variables, nueve campos completos por variable.


## Exportación determinista y recarga

In [7]:
df_final, sha256, resultados_finales = exportar_dataset_final(
    RUTA_CANDIDATO, RUTA_FINAL
)
validar_invariantes_finales(df_final)

assert set(df_final["DEPARTAMENTO"]) == set(catalogos.DEPARTAMENTOS_OFICIALES)
assert not df_final.isna().all(axis=1).any()
assert not df_final.duplicated().any()
assert df_final["CODIGO"].is_unique
assert list(df_final.columns) == COLUMNAS_REQUERIDAS
print("Salida:", RUTA_FINAL.relative_to(ROOT))
print("Dimensión:", df_final.shape)
print("Cobertura departamental:", df_final["DEPARTAMENTO"].nunique())
print("SHA-256:", sha256)

Salida: data/processed/establecimientos_clean.csv
Dimensión: (11868, 18)
Cobertura departamental: 22
SHA-256: 429468cb07a6e31aeb2a1d27611b0d82952a5c909f2b6a454fc03b12a1ec5b49


## Resultado final

In [8]:
resumen_final = resumen_validaciones(resultados_finales)
display(resumen_final)
if len(resultados_finales) != 7 or not all(r.aprobada for r in resultados_finales):
    raise RuntimeError("VALIDACIÓN FINAL FALLIDA: no se entrega el CSV.")

print("APROBADO — dataset final v1.0.0")
print("7 de 7 validaciones aprobadas; 11,868 filas; 18 columnas; 22 departamentos.")
print("Hash SHA-256 reproducible:", sha256)

,Prueba,Estado,Errores,Detalle
0,Duplicados exactos,APROBADO,0,No deben existir filas idénticas en todas las ...
1,Espacios en textos,APROBADO,0,"Los textos no deben tener espacios al inicio, ..."
2,Formato de teléfonos,APROBADO,0,"Cada teléfono debe tener 8 dígitos, iniciar en..."
3,Geografía oficial,APROBADO,0,Departamento y municipio deben pertenecer al c...
4,Esquema y tipos,APROBADO,0,El candidato debe contener las 18 columnas esp...
5,Categorías equivalentes,APROBADO,0,No deben coexistir categorías que solo difiera...
6,Valores inválidos diagnosticados,APROBADO,0,"Códigos, distritos, dominios, faltantes obliga..."


APROBADO — dataset final v1.0.0
7 de 7 validaciones aprobadas; 11,868 filas; 18 columnas; 22 departamentos.
Hash SHA-256 reproducible: 429468cb07a6e31aeb2a1d27611b0d82952a5c909f2b6a454fc03b12a1ec5b49


## Contrato de escritura

El CSV se escribe en UTF-8, sin índice, con salto de línea `LF`, faltantes como campos vacíos, columnas en el orden del esquema y filas ordenadas establemente por `CODIGO`. Todos los campos se recargan mediante el esquema `string`; el CSV por sí mismo no conserva tipos.